# Problem 1: Representing SHA-256 Data in Python

This notebook explains the choices I made for representing SHA-256 data, with a small demo for each one.

## Background

SHA-256 is defined in terms of 32-bit words, 512-bit message blocks and big-endian byte order. Python's built-in `int` has arbitrary precision, so it never overflows and has no fixed size. That doesn't match the standard on its own, so I looked at how to represent each piece of data. Because Problems 2 and 3 require NumPy, I use NumPy's fixed-width `uint32` type for words from the start, so the whole project uses one consistent convention.

## Summary

| Concept | Representation |
|---|---|
| 32-bit word | `np.uint32` |
| Sequence of 32-bit words | NumPy array with `dtype=np.uint32` |
| Input message | `bytes` |
| 512-bit message block | 64 bytes, read as an array of 16 `uint32` words |
| Final 256-bit hash | 8 `uint32` words, then 32 `bytes`, then 64 hex characters |
| Big-endian integers | `int.from_bytes(..., "big")` and the `">u4"` dtype |

In [4]:
import numpy as np
import hashlib

# 1. 32-bit words

# a normal python int just keeps growing
big = 0xFFFFFFFF + 1
print("python int:", hex(big))

# a uint32 wraps back to 0
x = np.array([0xFFFFFFFF], dtype=np.uint32)
y = np.array([1], dtype=np.uint32)
total = x + y
print("uint32:", hex(int(total[0])))

# rotate right (used a lot in sha-256)
def rotr(value, n):
    value = np.uint32(value)
    left = np.uint32(32 - n)
    n = np.uint32(n)
    return (value >> n) | (value << left)

print(hex(int(rotr(0x00000001, 1))))
print(hex(int(rotr(0x12345678, 8))))

python int: 0x100000000
uint32: 0x0
0x80000000
0x78123456


## 2. Sequences of 32-bit words

**Choice:** a NumPy array with `dtype=np.uint32`.

SHA-256 uses several fixed-length sequences of words: the 16-word message block, the 64-word message schedule, the 8 hash values, the 8 working variables and the 64 round constants. They are all indexed by position, so an array fits well, and every element keeps the 32-bit wraparound behaviour.

The `dtype` must always be written explicitly. Without it NumPy picks a default (usually 64-bit) and the 32-bit guarantee is lost. The code cell below builds the initial hash values and an empty 64-word schedule and checks their type and length.

### Problem 2: SHA-256 Bitwise Operations

### Problem 3: Generating the SHA-256 Constants

### Problem 4: Padding and Parsing Messages

### Problem 5: The SHA-256 Compression Function

### Problem 6: Complete SHA-256